# SVR Kernel Theory: Mathematics & Financial Application

This notebook goes deep on the kernel machinery that makes SVR powerful for financial time series.
It complements the main analysis notebook with visualisations of the key mathematical concepts.

**Topics covered:**
1. The kernel trick and Mercer's theorem
2. RBF, Linear, and Polynomial kernel geometry
3. Support vector visualisation
4. Bias-variance tradeoff through the lens of C and epsilon
5. VC dimension and generalisation in high-dimensional feature spaces
6. Practical kernel selection guide for finance

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
import seaborn as sns
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

sys.path.insert(0, '..')

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
SEED = 42
np.random.seed(SEED)
print('Ready')

---
## 1 · The Kernel Trick

The key insight: we never need to compute $\phi(\mathbf{x})$ explicitly. We only need inner products:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \langle \phi(\mathbf{x}_i),\, \phi(\mathbf{x}_j) \rangle_{\mathcal{H}}$$

**Mercer's Theorem**: A symmetric function $K: \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ is a valid kernel
(i.e., corresponds to an inner product in some RKHS) iff the Gram matrix $K_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$
is positive semi-definite for all finite sets $\{\mathbf{x}_1, \ldots, \mathbf{x}_n\}$.

**Practical consequence for finance**:
- RBF kernel: $K(\mathbf{x}_i, \mathbf{x}_j) = e^{-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2}$ — universal approximator; captures all smooth nonlinear patterns
- Linear kernel: $K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i^\top \mathbf{x}_j$ — equivalent to linear regression in the primal; interpretable factor loadings
- Polynomial: $(\gamma \mathbf{x}_i^\top \mathbf{x}_j + r)^d$ — explicit interaction effects up to degree $d$

In [ ]:
# Visualise RBF kernel as a function of distance
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1) RBF kernel as function of ||x_i - x_j||
dist = np.linspace(0, 5, 200)
for gamma, ls in [(0.1, '-'), (0.5, '--'), (2.0, ':')]:
    axes[0].plot(dist, np.exp(-gamma * dist**2), ls=ls, lw=2, label=f'gamma={gamma}')
axes[0].set_xlabel('||x_i - x_j||')
axes[0].set_ylabel('K(x_i, x_j)')
axes[0].set_title('RBF Kernel vs Distance', fontweight='bold')
axes[0].legend()
axes[0].annotate('Localised similarity:\nhigh gamma = narrow influence', xy=(1.5, 0.5),
                  xytext=(2.5, 0.8), fontsize=8, arrowprops=dict(arrowstyle='->'))

# 2) Gram matrix structure for financial time series
np.random.seed(SEED)
n = 50
times = np.arange(n)
# Simulate feature vectors that are correlated across time
X_sim = np.column_stack([np.sin(0.3 * times) + 0.2*np.random.randn(n),
                          np.cos(0.2 * times) + 0.2*np.random.randn(n)])
X_sim = StandardScaler().fit_transform(X_sim)

# Gram matrices
gram_rbf  = np.exp(-0.5 * np.sum((X_sim[:, None] - X_sim[None, :]) ** 2, axis=-1))
gram_lin  = X_sim @ X_sim.T
gram_lin  = (gram_lin - gram_lin.min()) / (gram_lin.max() - gram_lin.min())

for ax, gram, title in zip(
    axes[1:],
    [gram_rbf, gram_lin],
    ['RBF Gram Matrix\n(captures regime similarity)', 'Linear Gram Matrix\n(linear factor structure)']
):
    im = ax.imshow(gram, cmap='YlOrRd', aspect='auto')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Training observation')
    ax.set_ylabel('Training observation')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Kernel Structure: How SVR "sees" Similarity Between Observations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/k01_kernel_structure.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 2 · Epsilon-Insensitive Loss vs. MSE

SVR minimises the epsilon-insensitive loss:
$$L_\varepsilon(y, f(x)) = \max(0,\; |y - f(x)| - \varepsilon)$$

Contrast with squared loss (used by OLS/Ridge):
$$L_{\text{MSE}}(y, f(x)) = (y - f(x))^2$$

**Financial interpretation of epsilon**:
- In financial return prediction, predictions within $\varepsilon$ of zero are treated as "no signal" — the model abstains
- This is equivalent to an automatic **signal-quality filter**: only sufficiently strong directional predictions trigger trades
- Empirically: $\varepsilon \approx$ transaction cost threshold ensures we only trade when the expected profit exceeds costs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

error = np.linspace(-0.1, 0.1, 400)

# Loss functions
for eps, color in [(0.005, PALETTE[0]), (0.01, PALETTE[1]), (0.02, PALETTE[2])]:
    eps_loss = np.maximum(0, np.abs(error) - eps)
    axes[0].plot(error * 100, eps_loss, color=color, lw=2, label=f'eps-SVR (eps={eps:.3f})')

mse_loss = error ** 2 * 1000
axes[0].plot(error * 100, mse_loss, 'k--', lw=1.5, alpha=0.7, label='MSE (scaled)')
axes[0].axvline(0, color='grey', lw=0.5)
axes[0].set_xlabel('Prediction Error (%)')
axes[0].set_ylabel('Loss')
axes[0].set_title('Epsilon-Insensitive vs MSE Loss', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].annotate('Zero-loss tube:\nno penalty for small errors', xy=(0, 0), xytext=(3, 0.008),
                  fontsize=9, arrowprops=dict(arrowstyle='->', color='red'), color='red')

# Effect of epsilon on number of support vectors
np.random.seed(SEED)
n_points = 200
X_1d = np.sort(np.random.uniform(0, 1, n_points)).reshape(-1, 1)
y_1d = np.sin(3 * np.pi * X_1d.ravel()) + 0.15 * np.random.randn(n_points)

epsilon_vals = [0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
n_sv = []
for eps in epsilon_vals:
    m = SVR(kernel='rbf', C=1.0, epsilon=eps, gamma='scale')
    m.fit(X_1d, y_1d)
    n_sv.append(m.support_vectors_.shape[0])

axes[1].plot(epsilon_vals, n_sv, 'o-', color=PALETTE[0], lw=2, ms=8)
axes[1].set_xlabel('Epsilon')
axes[1].set_ylabel('Number of Support Vectors')
axes[1].set_title('Epsilon -> Support Vectors\n(larger eps = simpler model)', fontweight='bold')
for eps, nsv in zip(epsilon_vals, n_sv):
    axes[1].annotate(f'{nsv}', (eps, nsv), textcoords='offset points',
                      xytext=(5, 5), fontsize=8)

plt.suptitle('The epsilon Parameter: Signal Filter & Model Complexity Control', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/k02_epsilon_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

print('Key insight: increasing epsilon reduces the number of support vectors,')
print('yielding a sparser, simpler model with fewer parameters to estimate.')
print('In finance: only trades when |prediction| > epsilon (acts as a conviction threshold).')

---
## 3 · Regularisation Parameter C: Bias-Variance Tradeoff

C controls how much we penalise observations outside the epsilon-tube:

$$\min_{\mathbf{w}, b, \xi} \underbrace{\frac{1}{2}\|\mathbf{w}\|^2}_{\text{model complexity}} + \underbrace{C \sum_i \xi_i}_{\text{training error}}$$

- **C large** ($C \to \infty$): zero tolerance for training errors → overfits, high variance
- **C small** ($C \to 0$): ignores training errors → underfits, high bias

In the **dual form**, C directly bounds the Lagrange multipliers: $0 \leq \alpha_i, \alpha_i^* \leq C$,
corresponding to how much influence any single training point can have on the decision function.

In [ ]:
# Simulate a financial-like noisy signal
np.random.seed(SEED)
n_train, n_test = 150, 50

# Generate 'true' underlying signal + noise
t_all = np.linspace(0, 4 * np.pi, n_train + n_test)
trend = 0.3 * t_all
signal_true = np.sin(t_all) + 0.5 * np.sin(2 * t_all) + trend
# Add heteroscedastic noise (like financial returns)
noise = np.random.randn(len(t_all)) * (0.3 + 0.3 * np.abs(np.sin(t_all)))
signal_noisy = signal_true + noise

X_ts  = t_all.reshape(-1, 1)
y_ts  = signal_noisy
X_tr  = X_ts[:n_train]; y_tr = y_ts[:n_train]
X_te  = X_ts[n_train:]; y_te = y_ts[n_train:]

C_vals = [0.01, 0.1, 1, 10, 100]
fig, axes = plt.subplots(1, len(C_vals), figsize=(16, 4), sharey=True)

for ax, C in zip(axes, C_vals):
    m = SVR(kernel='rbf', C=C, epsilon=0.05, gamma='scale')
    m.fit(X_tr, y_tr)
    pred_tr = m.predict(X_tr)
    pred_te = m.predict(X_te)
    
    rmse_tr = np.sqrt(mean_squared_error(y_tr, pred_tr))
    rmse_te = np.sqrt(mean_squared_error(y_te, pred_te))
    
    ax.scatter(X_tr.ravel(), y_tr, s=5, alpha=0.4, color=PALETTE[0], label='Train')
    ax.scatter(X_te.ravel(), y_te, s=5, alpha=0.4, color=PALETTE[1], label='Test')
    ax.plot(X_ts.ravel(), m.predict(X_ts), color='red', lw=2, label='SVR fit')
    
    # Mark support vectors
    sv_idx = m.support_
    ax.scatter(X_tr[sv_idx].ravel(), y_tr[sv_idx], s=40, facecolors='none',
               edgecolors='green', lw=1.5, label='Support Vecs')
    
    ax.set_title(f'C = {C}\nTrain RMSE={rmse_tr:.3f}\nTest RMSE={rmse_te:.3f}', fontweight='bold')
    ax.set_xlabel('t')
    ax.annotate(f'#SV={m.support_vectors_.shape[0]}', xy=(0.05, 0.95),
                xycoords='axes fraction', fontsize=9, color='green', fontweight='bold')

axes[0].set_ylabel('Signal')
axes[0].legend(fontsize=7, loc='upper left')
plt.suptitle('Effect of C on Model Complexity and Generalisation (epsilon=0.05, RBF kernel)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/k03_C_regularisation.png', bbox_inches='tight', dpi=150)
plt.show()

print('Observation: small C = many support vectors (simple model); large C = few support vectors (complex fit).')
print('Optimal C balances train RMSE against test RMSE (bias-variance tradeoff).')

---
## 4 · Kernel Bandwidth (gamma) and Effective Complexity

For the RBF kernel $K(x_i, x_j) = e^{-\gamma \|x_i - x_j\|^2}$, the bandwidth $1/\sqrt{2\gamma}$
controls the **effective radius of influence** of each support vector:

- **Large gamma**: narrow Gaussians — each SV only influences its immediate neighbourhood → overfitting
- **Small gamma**: wide Gaussians — global influence → underfitting

In scikit-learn: `gamma='scale'` sets $\gamma = 1 / (d \cdot \text{Var}(X))$ and is the recommended default.

In [ ]:
gamma_vals = [0.01, 0.1, 1.0, 10.0, 100.0]
fig, axes = plt.subplots(1, len(gamma_vals), figsize=(16, 4), sharey=True)

for ax, gamma in zip(axes, gamma_vals):
    m = SVR(kernel='rbf', C=10, epsilon=0.05, gamma=gamma)
    m.fit(X_tr, y_tr)
    pred_tr = m.predict(X_tr)
    pred_te = m.predict(X_te)
    
    rmse_te = np.sqrt(mean_squared_error(y_te, pred_te))
    
    ax.scatter(X_tr.ravel(), y_tr, s=5, alpha=0.3, color=PALETTE[0])
    ax.scatter(X_te.ravel(), y_te, s=5, alpha=0.3, color=PALETTE[1])
    ax.plot(X_ts.ravel(), m.predict(X_ts), color='red', lw=2)
    ax.set_title(f'gamma = {gamma}\nTest RMSE={rmse_te:.3f}\n#SV={m.support_vectors_.shape[0]}',
                 fontweight='bold')
    ax.set_xlabel('t')

axes[0].set_ylabel('Signal')
plt.suptitle('Effect of gamma (RBF bandwidth) on Fit Complexity  |  C=10, epsilon=0.05',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/k04_gamma_bandwidth.png', bbox_inches='tight', dpi=150)
plt.show()

print('gamma too small: underfit (ignores local structure)')
print('gamma too large: overfit (memorises noise, poor generalisation)')
print('gamma="scale": data-adaptive choice recommended as starting point')

---
## 5 · TimeSeriesSplit: Why Random K-Fold Fails in Finance

Standard k-fold cross-validation **leaks future information** into training sets in time series:

```
Random k-fold (WRONG for time series):
  Fold 1: test=[2,5,9], train=[1,3,4,6,7,8,10]  <- future obs in train!

TimeSeriesSplit (CORRECT):
  Fold 1: train=[1,2,3], test=[4,5]
  Fold 2: train=[1,2,3,4,5], test=[6,7]
  ...
```

**Why this matters**: A model trained on tomorrow's data will always look better out-of-sample than it truly is. This is the most common source of backtest overfitting in quantitative research.

In [ ]:
# Visualise TimeSeriesSplit vs random k-fold
from sklearn.model_selection import KFold

n_samples = 20
np.random.seed(SEED)
X_demo = np.arange(n_samples).reshape(-1, 1)
y_demo = np.random.randn(n_samples)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

for ax, splitter, title in zip(
    axes,
    [TimeSeriesSplit(n_splits=5), KFold(n_splits=5, shuffle=True, random_state=SEED)],
    ['TimeSeriesSplit (CORRECT for financial data)', 'Random KFold (WRONG - leaks future data)']
):
    ax.set_title(title, fontweight='bold', color='green' if 'CORRECT' in title else 'red')
    ax.set_xlim(-1, n_samples)
    ax.set_ylim(-1, 5.5)
    ax.set_xlabel('Observation index (time)')
    ax.set_ylabel('CV Fold')
    ax.set_yticks(range(5)); ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
    ax.axvline(14, color='grey', ls=':', lw=1, label='Present')
    
    for fold, (tr_idx, te_idx) in enumerate(splitter.split(X_demo, y_demo)):
        ax.barh(fold, len(tr_idx), left=tr_idx.min(), height=0.5,
                color=PALETTE[0], alpha=0.6, label='Train' if fold == 0 else '')
        ax.barh(fold, len(te_idx), left=te_idx.min(), height=0.5,
                color=PALETTE[3], alpha=0.8, label='Test' if fold == 0 else '')
        # Highlight future-in-train problem for KFold
        if 'KFold' in title.__class__.__name__ or 'Random' in title:
            max_test = te_idx.max()
            future_train = tr_idx[tr_idx > max_test]
            if len(future_train) > 0:
                ax.scatter(future_train, [fold] * len(future_train), marker='x',
                           color='red', s=80, zorder=5, label='Future in train!' if fold == 0 else '')
    ax.legend(fontsize=9, loc='upper left')

plt.suptitle('Cross-Validation Strategy Comparison: TimeSeriesSplit vs KFold', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/k05_cv_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 6 · Kernel Selection Guide for Quantitative Finance

### Decision Framework

```
                    Is the signal likely linear?
                   /                           \
                 YES                            NO
                  |                              |
           Linear kernel               Are feature interactions
         (interpretable,                    important?
          fast to train)                 /           \
                                       YES             NO
                                        |               |
                                   Polynomial       RBF kernel
                                   (explicit      (universal,
                                  interactions)    smooth fit)
```

### When to Use Each Kernel in Finance

| Kernel | Best for | Avoid when |
|---|---|---|
| **RBF** | Volatility prediction, regime detection, default choice | Feature space is truly linear (waste of complexity) |
| **Linear** | Factor models, cross-sectional ranking, n >> d | Strong nonlinear regime effects |
| **Polynomial (d=2)** | Momentum x volatility interactions | d > 3 (overfits badly in financial data) |

### Computational Complexity

| Kernel | Training | Prediction |
|---|---|---|
| Linear | O(n * d) | O(n_sv * d) |
| RBF | O(n^2 * d) | O(n_sv * d) |
| Polynomial | O(n^2 * d) | O(n_sv * d) |

In [ ]:
def _load_csv(path):
    """Read yfinance CSV, skipping the Ticker header row if present."""
    import pandas as pd
    raw = pd.read_csv(path, index_col=0, header=0)
    if raw.index[0] == 'Ticker' or str(raw.iloc[0, 0]) in ('AAPL', 'MSFT', 'SPY'):
        raw = raw.iloc[1:]
    raw = raw.apply(pd.to_numeric, errors='coerce').dropna(how='all')
    raw.index = pd.to_datetime(raw.index)
    raw.index.name = 'Date'
    return raw
# Empirical kernel comparison: load real AAPL data and compare kernels on return prediction
import os

DATA_PATH = '../data/AAPL_2015-01-01_2024-12-31.csv'
if os.path.exists(DATA_PATH):
    data = _load_csv(DATA_PATH)
    if 'Returns' not in data.columns:
        data['Returns'] = data['Close'].pct_change()
    data = data.dropna()
    print(f'Loaded {len(data):,} days of AAPL data')
else:
    print('Data file not found; run main notebook first to download')
    data = None

if data is not None:
    # Simple lagged features for quick kernel demo
    lags = 10
    df_feat = pd.DataFrame({'y': data['Returns'].shift(-1)})
    for lag in range(1, lags + 1):
        df_feat[f'r_lag{lag}'] = data['Returns'].shift(lag)
    df_feat['vol5']  = data['Returns'].rolling(5).std()
    df_feat['vol20'] = data['Returns'].rolling(20).std()
    df_feat = df_feat.dropna()

    X_k  = df_feat.drop(columns='y').values
    y_k  = df_feat['y'].values
    split = int(0.7 * len(X_k))
    X_tr_k, X_te_k = X_k[:split], X_k[split:]
    y_tr_k, y_te_k = y_k[:split], y_k[split:]
    sc = StandardScaler()
    X_tr_ks = sc.fit_transform(X_tr_k)
    X_te_ks = sc.transform(X_te_k)

    results = {}
    for name, kern, C, gamma in [
        ('RBF',        'rbf',    100, 'scale'),
        ('Linear',     'linear', 10,  'scale'),
        ('Poly d=2',   'poly',   100, 'scale'),
        ('Poly d=3',   'poly',   100, 'scale'),
    ]:
        kw = dict(kernel=kern, C=C, epsilon=0.005, gamma=gamma)
        if kern == 'poly':
            kw['degree'] = int(name[-1])
        m = SVR(**kw)
        m.fit(X_tr_ks, y_tr_k)
        p = m.predict(X_te_ks)
        da = (np.sign(y_te_k) == np.sign(p)).mean()
        rmse = np.sqrt(mean_squared_error(y_te_k, p))
        results[name] = {'DA': da, 'RMSE': rmse, 'n_SV': m.support_vectors_.shape[0]}
        print(f'  {name:<12}: DA={da:.4f}  RMSE={rmse:.6f}  n_SV={m.support_vectors_.shape[0]}')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    names = list(results.keys())
    das   = [results[n]['DA'] for n in names]
    rmses = [results[n]['RMSE'] for n in names]
    
    axes[0].bar(names, das, color=['#1f77b4','#ff7f0e','#2ca02c','#d62728'], alpha=0.8)
    axes[0].axhline(0.5, color='black', ls='--', lw=1, label='Random (50%)')
    axes[0].set_ylabel('Directional Accuracy'); axes[0].set_ylim(0.45, 0.6)
    axes[0].set_title('Directional Accuracy by Kernel', fontweight='bold')
    axes[0].legend()
    for i, (bar, val) in enumerate(zip(axes[0].patches, das)):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    
    axes[1].bar(names, rmses, color=['#1f77b4','#ff7f0e','#2ca02c','#d62728'], alpha=0.8)
    axes[1].set_ylabel('RMSE')
    axes[1].set_title('RMSE by Kernel (lower = better level prediction)', fontweight='bold')
    for i, (bar, val) in enumerate(zip(axes[1].patches, rmses)):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                     f'{val:.5f}', ha='center', va='bottom', fontsize=8)
    
    plt.suptitle('Empirical Kernel Comparison on AAPL Return Prediction (OOS)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/k06_empirical_kernel_comparison.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## 7 · Summary: SVR for Quantitative Finance

### Why SVR Is Well-Suited for Financial ML

1. **Structural Risk Minimisation**: Controls model complexity through $\|\mathbf{w}\|^2$, not just training error. Critical when $n/d$ is small.

2. **Robust loss function**: The $\varepsilon$-insensitive loss is robust to small perturbations and acts as a built-in noise filter. MSE-based models overfit to return measurement noise.

3. **Sparsity**: Only support vectors determine the decision function. The model automatically ignores uninformative training points, which is valuable when many financial observations are pure noise.

4. **Kernel flexibility**: RBF can capture nonlinear regime dynamics; Linear corresponds to a factor model; both are principled choices depending on the hypothesis.

5. **Non-parametric**: No assumption of normality (unlike OLS). Heavy-tailed financial returns are handled gracefully.

### Limitations

1. **Scaling**: O(n^2) to O(n^3) training time. For very large datasets, consider online SVR (LASVM) or approximations (Nystroem features + linear SVM).

2. **Non-probabilistic**: SVR does not output probabilities. For position sizing based on prediction confidence, use prediction magnitude (|f(x)|) or calibrate via isotonic regression.

3. **Stationarity assumption**: Kernel choice and hyperparameters assume some stationarity. In non-stationary regimes, walk-forward re-estimation is essential.

4. **No uncertainty quantification**: Unlike Gaussian Processes (which also use kernel methods), SVR does not provide prediction intervals natively.